# Notebook 5 — Routing Questions to the Right Workflow

### Build an AI Business Copilot 🏔️ · Session 2

Your copilot now has two skills:

- 📄 **Documents (RAG)** — answers policy questions (Notebooks 2–3).
- 🗄️ **Data (tool use)** — answers order/inventory questions (Notebook 4).

But there's a catch: so far *you* choose which skill to use by calling the right function. A
real customer just types **one question** — and the copilot has to decide for itself which
skill to use.

In this notebook we add a **router**: a small step that reads the question and sends it to
the right skill. Then we wire everything into a single `ask_copilot()` function — the whole
copilot behind one door.

By the end you'll be able to:

1. Explain why a copilot needs a router.
2. Use Claude to **classify** a question as *policy*, *data*, or *both*.
3. Combine your two skills behind one `ask_copilot()` entry point.
4. Handle blended questions that need documents **and** data at once.

## 1. Setup

This notebook uses **both** skills, so we install the retrieval libraries *and* need the API
key. Run the three setup cells.

In [1]:
%pip install -q anthropic sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.8/999.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 31.6 MB/s eta 0:00:00


In [2]:
# --- Make the workshop files available (Colab, Databricks, or local) ---
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/allistaircota/trailhead-copilot"   # <-- ✏️ set to the real repo URL

def find_repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if all((base / d).is_dir() for d in ("data", "documents", "src")):
            return base
    cloned = Path.cwd() / "trailhead-copilot"
    return cloned if (cloned / "src").is_dir() else None

root = find_repo_root()
if root is None:
    print("Cloning the workshop repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    root = find_repo_root()
if root is None:
    raise RuntimeError("Workshop files not found — set REPO_URL to the real repository URL.")

sys.path.insert(0, str(root / "src"))
print("Using workshop files at:", root)

Cloning the workshop repo...
Using workshop files at: /content/trailhead-copilot


In [3]:
import os, getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")
print("API key is set. ✅")

Paste your Anthropic API key: ··········
API key is set. ✅


In [4]:
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 500
TOP_K = 4

## 2. Rebuild Skill 1 — the document Q&A (from Notebook 3)

A compact version of the RAG copilot you already built. `answer_from_documents()` retrieves
policy chunks and asks Claude to answer from them.

In [5]:
import trailhead as th

# Retriever (Notebook 2)
chunks = th.build_chunks()
index, embedder = th.build_index(chunks)

def retrieve(query, k=TOP_K):
    return th.search(query, index, chunks, embedder, top_k=k)

# Grounded answer (Notebook 3)
SYSTEM_DOCS = (
    "You are a support assistant for Trailhead Supply Co. Answer using ONLY the policy "
    "excerpts provided. If the answer isn't in them, say you don't have that information. "
    "Keep it short and cite the document like [source: return_policy.md]."
)

def build_context(hits):
    return "\n\n".join("[source: " + h["source"] + "]\n" + h["text"] for h in hits)

def answer_from_documents(query):
    hits = retrieve(query)
    prompt = "POLICY EXCERPTS:\n" + build_context(hits) + "\n\nCUSTOMER QUESTION: " + query
    return th.ask_claude(prompt, system=SYSTEM_DOCS, model=MODEL, max_tokens=MAX_TOKENS)

print(answer_from_documents("What is the return window for hiking boots?"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Hiking boots can be returned within the same **30-day window** as other items, measured from the delivery date. However, they must be **unworn outdoors** — trying them on indoors is fine to check fit, but items with visible outdoor wear (dirt, scuffs, trail damage) cannot be returned [source: return_policy.md].


## 3. Rebuild Skill 2 — the data lookups (from Notebook 4)

The order/inventory tools and the tool-use loop, packaged as `answer_from_data()`. Notice we
gave `run_with_tools()` a `system` argument this time — we'll reuse it for blended questions
in a moment.

In [6]:
import anthropic, json
import pandas as pd

data = th.load_data()
orders, shipments = data["orders"], data["shipments"]
products, inventory = data["products"], data["inventory"]
client = anthropic.Anthropic()

def clean(v):
    return None if pd.isna(v) else v

def get_order_status(order_id):
    order_id = int(order_id)
    o = orders[orders["order_id"] == order_id]
    if o.empty:
        return {"error": "No order found with ID " + str(order_id)}
    o = o.iloc[0]
    r = {"order_id": order_id, "order_status": clean(o["status"]),
         "shipping_method": clean(o["shipping_method"]), "order_total": float(o["total"])}
    s = shipments[shipments["order_id"] == order_id]
    if not s.empty:
        s = s.iloc[0]
        r["shipment_status"] = clean(s["status"])
        r["carrier"] = clean(s["carrier"])
        r["tracking_number"] = clean(s["tracking_number"])
        r["estimated_delivery"] = clean(s["estimated_delivery"])
        r["delivered_date"] = clean(s["delivered_date"])
    return r

def check_inventory(product_name):
    m = products[products["name"].str.contains(product_name, case=False, na=False)]
    if m.empty:
        return {"error": "No product matching '" + product_name + "'."}
    found = []
    for _, p in m.iterrows():
        inv = inventory[inventory["product_id"] == p["product_id"]]
        units = int(inv.iloc[0]["units_in_stock"]) if not inv.empty else 0
        found.append({"product": p["name"], "units_in_stock": units, "in_stock": units > 0})
    return {"matches": found}

tools = [
    {"name": "get_order_status",
     "description": "Look up status, shipping method, carrier, tracking, and estimated delivery for an order by numeric ID.",
     "input_schema": {"type": "object",
                      "properties": {"order_id": {"type": "integer", "description": "Numeric order ID, e.g. 1001"}},
                      "required": ["order_id"]}},
    {"name": "check_inventory",
     "description": "Check units in stock for a product by full or partial name.",
     "input_schema": {"type": "object",
                      "properties": {"product_name": {"type": "string", "description": "Full or partial product name"}},
                      "required": ["product_name"]}},
]
TOOL_FUNCTIONS = {"get_order_status": get_order_status, "check_inventory": check_inventory}

SYSTEM_DATA = (
    "You are a support assistant for Trailhead Supply Co. Use the tools to look up real "
    "order and inventory data. Only state facts returned by a tool; never guess. Keep replies brief."
)

def run_with_tools(user_message, system):
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=MAX_TOKENS, system=system, tools=tools, messages=messages)
        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text")
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                out = TOOL_FUNCTIONS[block.name](**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(out)})
        messages.append({"role": "user", "content": results})

def answer_from_data(query):
    return run_with_tools(query, SYSTEM_DATA)

print(answer_from_data("Where is order 1001?"))

Order 1001 is **in transit** via UPS Expedited shipping. 

- **Tracking number:** 1Z8293453178
- **Estimated delivery:** January 6, 2026


## 4. The router — deciding which skill to use

The router is a tiny, cheap Claude call whose only job is to **classify** the question into
one of three buckets:

- **`policy`** — return/shipping/warranty rules, membership, gear care → the documents.
- **`data`** — a specific order, shipment, or stock level → the tools.
- **`both`** — needs a policy rule *and* a specific record (e.g., *"I want to return order
  1007 — how long do I have?"*).

We ask Claude to reply with just one word, then read that word. (Capping `max_tokens` at a
few keeps this essentially free.)

In [7]:
ROUTER_SYSTEM = (
    "You classify a customer question into exactly one category for a support copilot:\n"
    "- policy: about return, shipping, or warranty rules, membership, or gear care.\n"
    "- data: about a specific order, shipment, or product's stock level.\n"
    "- both: needs a policy rule AND a specific order/product lookup.\n"
    "Reply with ONLY one word: policy, data, or both."
)

def route(question):
    label = th.ask_claude(question, system=ROUTER_SYSTEM, model=MODEL, max_tokens=5)
    label = label.strip().lower()
    # Check 'both' first so it isn't shadowed by the words 'policy'/'data'.
    for category in ("both", "policy", "data"):
        if category in label:
            return category
    return "both"   # safe fallback if the reply is unexpected

for q in [
    "What is your return policy on final sale items?",
    "Where is order 1003?",
    "Is the Blaze stove in stock?",
    "I want to return order 1007 — how long do I have?",
]:
    print(f"{route(q):7}  <-  {q}")

policy   <-  What is your return policy on final sale items?
data     <-  Where is order 1003?
data     <-  Is the Blaze stove in stock?
both     <-  I want to return order 1007 — how long do I have?


### Why classify with Claude instead of keywords?

You might be tempted to route with simple rules — *"if the question contains 'return', use
documents."* But keywords are brittle: a customer might ask *"can I send these boots back?"*
(no "return") or *"where's my stuff?"* (no "order"). Just like the semantic search in
Notebook 2, an LLM classifier understands **meaning**, so it routes correctly no matter how
the customer phrases things.

## 5. Assemble the copilot — `ask_copilot()`

Now we put it together: **route** the question, then call the matching skill. For the
`both` case, we hand Claude the retrieved policy excerpts *and* the data tools in one call,
so it can combine them.

In [8]:
SYSTEM_BOTH = (
    "You are a support assistant for Trailhead Supply Co. Use the policy excerpts in the "
    "message for policy rules, and use the tools to look up the specific order or product. "
    "Combine both to give one clear answer. Only state facts from the excerpts or the tools."
)

def answer_both(question):
    hits = retrieve(question)
    prompt = "POLICY EXCERPTS:\n" + build_context(hits) + "\n\nCUSTOMER QUESTION: " + question
    return run_with_tools(prompt, SYSTEM_BOTH)

def ask_copilot(question, verbose=True):
    category = route(question)
    if verbose:
        print(f"[router -> {category}]")
    if category == "policy":
        return answer_from_documents(question)
    if category == "data":
        return answer_from_data(question)
    return answer_both(question)

# One door, any question:
print(ask_copilot("How long do I have to return a final sale item?"))
print()
print(ask_copilot("Is the Voyager 65L Backpack in stock?"))

[router -> policy]
Final Sale items are **not returnable or refundable** — there is no return window for them [source: return_policy.md].

You can check whether an item is marked Final Sale on its product page or in your order confirmation. 

If you believe the item is actually **defective or damaged**, that would be covered by our Trailhead Guarantee instead. Let me know and I can help you with that!

[router -> data]
Yes, the Voyager 65L Backpack is in stock with 70 units available.


## 6. The blended case — documents *and* data together

This is where it all comes together. *"I want to return order 1007 — how long do I have?"*
needs the **return policy** (30 days from delivery) *and* order 1007's **delivery date**
(from the database). Watch the copilot use both skills in one answer.

In [9]:
print(ask_copilot("I want to return order 1007 — how long do I have?"))

[router -> both]
Great! Your order 1007 was delivered on **January 6, 2026**.

Here's how long you have to return items:

- **Standard return window: 30 days** from delivery for unused items in original packaging with tags attached
- **Extended window: 60 days** if you're a member of our free **Trail Club** loyalty program

Based on your delivery date of January 6, your **30-day window closes on February 5, 2026**. If you're a Trail Club member, you'll have until **March 7, 2026**.

**Please note:** Items marked **Final Sale** (clearance or heavily discounted products) are not returnable. You can check your order confirmation to see if any items have this status.

To start your return, you can:
1. Sign in and open **Order History** to select the item and generate a prepaid return label, or
2. Contact our support team at **support@trailheadsupply.example** (Mon–Fri, 8am–6pm Mountain Time) or use live chat on our website.

Once you have your label, drop the package at any carrier locatio

The copilot retrieved the return policy, looked up order 1007's delivery date with a tool,
and combined them into one grounded answer. That's the full architecture from Notebook 1,
working end to end. 🎉

> ✏️ **Try it.** Ask a mix of questions and watch the `[router -> ...]` tag show how each one
> is handled. Try to phrase questions in unexpected ways to test the router.

In [10]:
for q in [
    "Can I send these boots back if I've worn them?",
    "Where's my stuff for order 1005?",
    "Do you have any Traverse boots left, and what's the warranty on them?",
]:
    print("Q:", q)
    print("A:", ask_copilot(q, verbose=True))
    print("-" * 70)

Q: Can I send these boots back if I've worn them?
[router -> policy]
A: It depends on how they've been worn.

**If worn indoors only:** Yes, boots can be returned within 30 days of delivery as long as they're unworn outdoors. Trying them on indoors is fine for checking fit.

**If worn outdoors:** No, boots with visible outdoor wear (dirt, scuffs, trail damage) cannot be returned under our standard return policy.

However, if there's a **manufacturing defect** (like a seam failure or hardware defect), that's covered under our warranty regardless of wear—no 30-day cutoff applies. In that case, contact support with photos and your order number [source: support_playbook.md, warranty_policy.md].
----------------------------------------------------------------------
Q: Where's my stuff for order 1005?
[router -> data]
A: Your order 1005 has shipped via **USPS Expedited**. Here are the details:

- **Tracking Number:** 1Z1240251661
- **Estimated Delivery:** January 16, 2026
- **Status:** Delay

## 7. What we built, and what's next

You now have a **complete copilot**: one `ask_copilot()` function that routes any question to
the right skill — documents, data, or both — and answers it with grounded facts. Every box
in the Notebook 1 architecture diagram is now real.

But "it works on the questions I tried" isn't good enough for something customers rely on.
**Notebook 6** is about making it trustworthy: we'll **evaluate** the copilot on a set of
questions, spot where it slips, **harden** the guardrails, and look at **cost controls** —
then wrap up with ideas for extending the copilot to your own use case.

## 8. Recap

- A copilot needs a **router** to decide which skill answers each question.
- We classify with a tiny Claude call (`policy` / `data` / `both`) — robust to phrasing,
  just like semantic search.
- `ask_copilot()` routes, then calls the document skill, the data skill, or **both**.
- The `both` path hands Claude retrieved policy text *and* the data tools in one call, so it
  can blend them.

## 9. Exercises

**Guided**
1. Call `route()` on five questions of your own and check each label. Find one the router
   gets wrong (if you can) and think about why.
2. Add a friendly greeting: if `route()` returns something for a non-support message like
   "hello", how does the copilot respond? Try it.

**Challenge**
3. Add a fourth category, `chit_chat`, for greetings/thanks that need no lookup, and have
   `ask_copilot()` reply directly (no documents or tools) for that case. Update
   `ROUTER_SYSTEM` and `ask_copilot()`.
4. Make the router **auditable**: change `route()` to also return a one-sentence reason for
   its choice, and print it. (Hint: ask for a label *and* a short reason, then parse the
   first word.) Why is an auditable router valuable in production?

## 10. Learn more

- **Anthropic — Classification with Claude:** <https://docs.claude.com/en/docs/about-claude/use-case-guides/ticket-routing>
- **Anthropic — Building effective agents (routing patterns):** <https://www.anthropic.com/research/building-effective-agents>
- **Anthropic — Tool use overview:** <https://docs.claude.com/en/docs/agents-and-tools/tool-use/overview>
- **Codecademy — Intro to Generative AI:** <https://www.codecademy.com/learn/intro-to-generative-ai>